In [1]:
import os
# Check if we are running on Kaggle's servers
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    DATA_DIR = '/kaggle/input/competitions/march-machine-learning-mania-2026'
else:
    DATA_DIR = '.'

# SETUP

In [2]:
import warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, log_loss, accuracy_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_info_columns', 999)
#pd.set_option('display.max_info_rows', 999)

torch.manual_seed(923434)
np.random.seed(923434)

# â”€â”€ Configuration â”€â”€
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
#Hyperparameters
CFG = {
    'hidden_dim':      128,   # size of GNN embeddings
    'num_layers':      3,     # number of GraphSAGE layers
    'dropout':         0.3,
    'lr':              1e-3,
    'weight_decay':    1e-4,
    'epochs':          150,
    'patience':        20,    # early stopping
    'val_season':      2025,  # season used for walk-forward validation
    'min_season':      2003,  # first seasons with complete data
    'label_smoothing': 0.05,  # avoids overconfident predictions (helps log loss)
}

print(f"Device: {DEVICE}")
print(f"Config: {CFG}")

Device: cuda
Config: {'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'epochs': 150, 'patience': 20, 'val_season': 2025, 'min_season': 2003, 'label_smoothing': 0.05}


# Load Data

In [4]:
def load_data(data_dir: str) -> dict:
    """Load all competition CSV files."""
    files = {
        'm_teams':   'MTeams.csv',
        'w_teams':   'WTeams.csv',
        'm_regular_detailed': 'MRegularSeasonDetailedResults.csv',
        'w_regular_detailed': 'WRegularSeasonDetailedResults.csv',
        'm_tourney_detailed': 'MNCAATourneyDetailedResults.csv',
        'w_tourney_detailed': 'WNCAATourneyDetailedResults.csv',
        'm_tourney': 'MNCAATourneyCompactResults.csv',
        'w_tourney': 'WNCAATourneyCompactResults.csv',
        'm_seeds':   'MNCAATourneySeeds.csv',
        'w_seeds':   'WNCAATourneySeeds.csv',
        'sample_sub':'SampleSubmissionStage1.csv',
        'sample_sub2':'SampleSubmissionStage2.csv'
    }

    data = {}
    for key, fname in files.items():
        path = f"{data_dir}/{fname}"
        try:
            data[key] = pd.read_csv(path)
            print(f"{fname}: {data[key].shape}")
        except FileNotFoundError:
            print(f"{fname} not found â€” skipping")
            data[key] = pd.DataFrame()

    return data

print("Loading data...")
DATA = load_data(DATA_DIR)

Loading data...
MTeams.csv: (381, 4)
WTeams.csv: (379, 2)
MRegularSeasonDetailedResults.csv: (124529, 34)
WRegularSeasonDetailedResults.csv: (87187, 34)
MNCAATourneyDetailedResults.csv: (1449, 34)
WNCAATourneyDetailedResults.csv: (961, 34)
MNCAATourneyCompactResults.csv: (2585, 8)
WNCAATourneyCompactResults.csv: (1717, 8)
MNCAATourneySeeds.csv: (2694, 3)
WNCAATourneySeeds.csv: (1812, 3)
SampleSubmissionStage1.csv: (519144, 2)
SampleSubmissionStage2.csv: (132133, 2)


# Data preparation

In [5]:
DATA['m_regular_detailed']

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124524,2026,132,1335,88,1463,84,N,1,29,64,14,28,16,16,11,26,17,13,5,4,16,31,71,9,24,13,17,17,24,23,9,9,4,16
124525,2026,132,1345,80,1276,72,N,0,30,57,4,14,16,22,7,22,21,2,4,5,13,30,64,7,24,5,6,11,22,15,7,2,3,18
124526,2026,132,1378,70,1455,55,N,0,23,61,6,22,18,23,14,27,13,10,10,3,20,19,56,4,19,13,20,12,27,8,14,7,7,19
124527,2026,132,1433,70,1173,62,N,0,23,57,11,24,13,19,6,28,12,5,9,4,18,23,53,7,21,9,17,8,31,16,11,3,4,18


In [6]:
DATA['w_regular_detailed']

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2010,11,3103,63,3237,49,H,0,23,54,5,9,12,19,10,26,14,18,7,0,15,20,54,3,13,6,10,11,27,11,23,7,6,19
1,2010,11,3104,73,3399,68,N,0,26,62,5,12,16,28,16,31,15,20,5,2,25,25,63,4,21,14,27,14,26,7,20,4,2,27
2,2010,11,3110,71,3224,59,A,0,29,62,6,15,7,12,14,23,18,13,6,2,17,19,58,2,14,19,23,17,23,8,15,6,0,15
3,2010,11,3111,63,3267,58,A,0,27,52,4,11,5,9,6,40,14,27,5,10,18,18,74,6,26,16,25,22,22,15,11,14,5,14
4,2010,11,3119,74,3447,70,H,1,30,74,7,20,7,11,14,33,18,11,5,3,18,25,74,9,17,11,21,21,32,12,14,4,2,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87182,2026,131,3471,60,3218,48,N,0,25,69,3,16,7,8,8,27,13,7,11,1,9,18,48,4,16,8,8,2,31,9,17,3,6,15
87183,2026,132,3158,68,3220,56,N,0,23,61,9,27,13,21,16,26,12,9,5,1,12,23,58,7,21,3,5,9,25,10,13,2,4,17
87184,2026,132,3192,79,3254,57,H,0,31,60,10,19,7,11,9,32,15,10,4,1,18,19,55,7,20,12,14,5,19,2,11,5,4,15
87185,2026,132,3221,77,3250,70,H,0,31,60,5,17,10,12,13,23,16,12,3,4,10,26,55,9,24,9,12,8,17,17,11,6,2,12


In [7]:
regular_results_detailed: pd.DataFrame = pd.concat([DATA['m_regular_detailed'], DATA['w_regular_detailed']], axis=0, ignore_index=True)
tourney_results_detailed: pd.DataFrame = pd.concat([DATA['m_tourney_detailed'], DATA['w_tourney_detailed']], axis=0, ignore_index=True)

regular_results_detailed


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211711,2026,131,3471,60,3218,48,N,0,25,69,3,16,7,8,8,27,13,7,11,1,9,18,48,4,16,8,8,2,31,9,17,3,6,15
211712,2026,132,3158,68,3220,56,N,0,23,61,9,27,13,21,16,26,12,9,5,1,12,23,58,7,21,3,5,9,25,10,13,2,4,17
211713,2026,132,3192,79,3254,57,H,0,31,60,10,19,7,11,9,32,15,10,4,1,18,19,55,7,20,12,14,5,19,2,11,5,4,15
211714,2026,132,3221,77,3250,70,H,0,31,60,5,17,10,12,13,23,16,12,3,4,10,26,55,9,24,9,12,8,17,17,11,6,2,12


In [8]:
regular_results_detailed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211716 entries, 0 to 211715
Data columns (total 34 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Season   211716 non-null  int64 
 1   DayNum   211716 non-null  int64 
 2   WTeamID  211716 non-null  int64 
 3   WScore   211716 non-null  int64 
 4   LTeamID  211716 non-null  int64 
 5   LScore   211716 non-null  int64 
 6   WLoc     211716 non-null  object
 7   NumOT    211716 non-null  int64 
 8   WFGM     211716 non-null  int64 
 9   WFGA     211716 non-null  int64 
 10  WFGM3    211716 non-null  int64 
 11  WFGA3    211716 non-null  int64 
 12  WFTM     211716 non-null  int64 
 13  WFTA     211716 non-null  int64 
 14  WOR      211716 non-null  int64 
 15  WDR      211716 non-null  int64 
 16  WAst     211716 non-null  int64 
 17  WTO      211716 non-null  int64 
 18  WStl     211716 non-null  int64 
 19  WBlk     211716 non-null  int64 
 20  WPF      211716 non-null  int64 
 21  LFGM     2

In [9]:
#mike kim fork
# Home is now Loc, deal with it later
# double the dataset with swapped team positions in box scores
def prepare_data(df: pd.DataFrame):
    df["LLoc"] = [{"H": "A", "A": "H", "N":"N"}[item] for item in df["WLoc"]]

    df = df[["Season", "DayNum", "LTeamID", "LScore", "WTeamID", "WScore", "NumOT",
            "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF", "LLoc",
            "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF", "WLoc"]]
    
    
    """
    # adjustment factor for overtimes, as more stats are accumulated during overtimes
    adjot = (40 + 5 * df["NumOT"]) / 40
    adjcols = ["LScore", "WScore", 
               "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
               "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]
    for col in adjcols:
        df[col] = df[col] / adjot
    """  
    #mapp = {'W': 'T1_', 'L': 'T2_'}  
    def label_swap(lab: str, mapp: dict) -> str:
        first = lab[0]
        if first in mapp:
            return mapp[first] + lab[1:]
        return lab
    
    #df.drop("WLoc", axis=1)
    dfswap = df.copy()
    df.columns = [label_swap(x, {'W': 'T1_', 'L': 'T2_'}) for x in list(df.columns)]
    dfswap.columns = [label_swap(x, {'L': 'T1_', 'W': 'T2_'}) for x in list(dfswap.columns)]
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output["PointDiff"] = output["T1_Score"] - output["T2_Score"]
    output["win"] = (output["PointDiff"] > 0) * 1
    output["men_women"] = (output["T1_TeamID"].apply(lambda t: str(t).startswith("1"))) * 1  # 0: women, 1: men
    return output

regular_results_detailed = prepare_data(regular_results_detailed)
tourney_results_detailed = prepare_data(tourney_results_detailed)

regular_results_detailed

,Season,DayNum,T2_TeamID,T2_Score,T1_TeamID,T1_Score,NumOT,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_Loc,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_Loc,PointDiff,win,men_women
0,2003,10,1328,62,1104,68,0,22,53,2,10,16,22,10,22,8,18,9,2,20,N,27,58,3,14,11,18,14,24,13,23,7,1,22,N,6,1,1
1,2003,10,1393,63,1272,70,0,24,67,6,24,9,20,20,25,7,12,8,6,16,N,26,62,8,20,10,19,15,28,16,13,4,4,18,N,7,1,1
2,2003,11,1437,61,1266,73,0,22,73,3,26,14,23,31,22,9,12,2,5,23,N,24,58,8,18,17,29,17,26,15,10,5,2,25,N,12,1,1
3,2003,11,1457,50,1296,56,0,18,49,6,22,8,15,17,20,9,19,4,3,23,N,18,38,3,9,17,31,6,19,11,12,14,2,18,N,6,1,1
4,2003,11,1208,71,1400,77,0,24,62,6,16,17,27,21,15,12,10,7,1,14,N,30,61,6,14,11,13,17,22,12,14,4,4,20,N,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423427,2026,131,3471,60,3218,48,0,25,69,3,16,7,8,8,27,13,7,11,1,9,N,18,48,4,16,8,8,2,31,9,17,3,6,15,N,-12,0,0
423428,2026,132,3158,68,3220,56,0,23,61,9,27,13,21,16,26,12,9,5,1,12,N,23,58,7,21,3,5,9,25,10,13,2,4,17,N,-12,0,0
423429,2026,132,3192,79,3254,57,0,31,60,10,19,7,11,9,32,15,10,4,1,18,H,19,55,7,20,12,14,5,19,2,11,5,4,15,A,-22,0,0
423430,2026,132,3221,77,3250,70,0,31,60,5,17,10,12,13,23,16,12,3,4,10,H,26,55,9,24,9,12,8,17,17,11,6,2,12,A,-7,0,0


In [10]:
regular_results_detailed.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423432 entries, 0 to 423431
Data columns (total 38 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Season     423432 non-null  int64 
 1   DayNum     423432 non-null  int64 
 2   T2_TeamID  423432 non-null  int64 
 3   T2_Score   423432 non-null  int64 
 4   T1_TeamID  423432 non-null  int64 
 5   T1_Score   423432 non-null  int64 
 6   NumOT      423432 non-null  int64 
 7   T2_FGM     423432 non-null  int64 
 8   T2_FGA     423432 non-null  int64 
 9   T2_FGM3    423432 non-null  int64 
 10  T2_FGA3    423432 non-null  int64 
 11  T2_FTM     423432 non-null  int64 
 12  T2_FTA     423432 non-null  int64 
 13  T2_OR      423432 non-null  int64 
 14  T2_DR      423432 non-null  int64 
 15  T2_Ast     423432 non-null  int64 
 16  T2_TO      423432 non-null  int64 
 17  T2_Stl     423432 non-null  int64 
 18  T2_Blk     423432 non-null  int64 
 19  T2_PF      423432 non-null  int64 
 20  T2_L

In [11]:
seeds: pd.DataFrame = pd.concat([DATA['m_seeds'], DATA['w_seeds']], axis=0, ignore_index=True)
seeds

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
4501,2026,Z12,3211
4502,2026,Z13,3453
4503,2026,Z14,3158
4504,2026,Z15,3239


In [12]:
seeds["seed"] = seeds["Seed"].apply(lambda x: int(x[1:3]))
seeds

,Season,Seed,TeamID,seed
0,1985,W01,1207,1
1,1985,W02,1210,2
2,1985,W03,1228,3
3,1985,W04,1260,4
4,1985,W05,1374,5
...,...,...,...,...
4501,2026,Z12,3211,12
4502,2026,Z13,3453,13
4503,2026,Z14,3158,14
4504,2026,Z15,3239,15


In [13]:
def seed_transform(df: pd.DataFrame) -> pd.DataFrame:
    seeds_T1 = seeds[["Season", "TeamID", "seed"]].copy()
    seeds_T2 = seeds[["Season", "TeamID", "seed"]].copy()
    seeds_T1.columns = ["Season", "T1_TeamID", "T1_seed"]
    seeds_T2.columns = ["Season", "T2_TeamID", "T2_seed"]

    df = pd.merge(df, seeds_T1, on=["Season", "T1_TeamID"], how="left")
    df = pd.merge(df, seeds_T2, on=["Season", "T2_TeamID"], how="left")

    # --- THE FIX ---
    # Assign a "ghost seed" to teams that didn't make the tournament
    # Seed 20 ensures they are mathematically treated as weaker than a 16-seed
    UNSEEDED_VAL = 20
    df["T1_seed"] = df["T1_seed"].fillna(UNSEEDED_VAL)
    df["T2_seed"] = df["T2_seed"].fillna(UNSEEDED_VAL)
    
    df["Seed_diff"] = df["T2_seed"] - df["T1_seed"]

    return df

tourney_results_detailed = seed_transform(tourney_results_detailed)
regular_results_detailed = seed_transform(regular_results_detailed)

regular_results_detailed

,Season,DayNum,T2_TeamID,T2_Score,T1_TeamID,T1_Score,NumOT,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_Loc,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_Loc,PointDiff,win,men_women,T1_seed,T2_seed,Seed_diff
0,2003,10,1328,62,1104,68,0,22,53,2,10,16,22,10,22,8,18,9,2,20,N,27,58,3,14,11,18,14,24,13,23,7,1,22,N,6,1,1,10.0,1.0,-9.0
1,2003,10,1393,63,1272,70,0,24,67,6,24,9,20,20,25,7,12,8,6,16,N,26,62,8,20,10,19,15,28,16,13,4,4,18,N,7,1,1,7.0,3.0,-4.0
2,2003,11,1437,61,1266,73,0,22,73,3,26,14,23,31,22,9,12,2,5,23,N,24,58,8,18,17,29,17,26,15,10,5,2,25,N,12,1,1,3.0,20.0,17.0
3,2003,11,1457,50,1296,56,0,18,49,6,22,8,15,17,20,9,19,4,3,23,N,18,38,3,9,17,31,6,19,11,12,14,2,18,N,6,1,1,20.0,20.0,0.0
4,2003,11,1208,71,1400,77,0,24,62,6,16,17,27,21,15,12,10,7,1,14,N,30,61,6,14,11,13,17,22,12,14,4,4,20,N,6,1,1,1.0,20.0,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423427,2026,131,3471,60,3218,48,0,25,69,3,16,7,8,8,27,13,7,11,1,9,N,18,48,4,16,8,8,2,31,9,17,3,6,15,N,-12,0,0,20.0,14.0,-6.0
423428,2026,132,3158,68,3220,56,0,23,61,9,27,13,21,16,26,12,9,5,1,12,N,23,58,7,21,3,5,9,25,10,13,2,4,17,N,-12,0,0,20.0,14.0,-6.0
423429,2026,132,3192,79,3254,57,0,31,60,10,19,7,11,9,32,15,10,4,1,18,H,19,55,7,20,12,14,5,19,2,11,5,4,15,A,-22,0,0,20.0,15.0,-5.0
423430,2026,132,3221,77,3250,70,0,31,60,5,17,10,12,13,23,16,12,3,4,10,H,26,55,9,24,9,12,8,17,17,11,6,2,12,A,-7,0,0,20.0,15.0,-5.0


In [14]:
regular_results_detailed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423432 entries, 0 to 423431
Data columns (total 41 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Season     423432 non-null  int64  
 1   DayNum     423432 non-null  int64  
 2   T2_TeamID  423432 non-null  int64  
 3   T2_Score   423432 non-null  int64  
 4   T1_TeamID  423432 non-null  int64  
 5   T1_Score   423432 non-null  int64  
 6   NumOT      423432 non-null  int64  
 7   T2_FGM     423432 non-null  int64  
 8   T2_FGA     423432 non-null  int64  
 9   T2_FGM3    423432 non-null  int64  
 10  T2_FGA3    423432 non-null  int64  
 11  T2_FTM     423432 non-null  int64  
 12  T2_FTA     423432 non-null  int64  
 13  T2_OR      423432 non-null  int64  
 14  T2_DR      423432 non-null  int64  
 15  T2_Ast     423432 non-null  int64  
 16  T2_TO      423432 non-null  int64  
 17  T2_Stl     423432 non-null  int64  
 18  T2_Blk     423432 non-null  int64  
 19  T2_PF      423432 non-n

In [15]:
boxcols = [
    "T1_Score", "T1_FGM", "T1_FGA", "T1_FGM3", "T1_FGA3", "T1_FTM", "T1_FTA",
    "T1_OR", "T1_DR", "T1_Ast", "T1_TO", "T1_Stl", "T1_Blk", "T1_PF",
    "T2_Score", "T2_FGM", "T2_FGA", "T2_FGM3", "T2_FGA3", "T2_FTM", "T2_FTA",
    "T2_OR", "T2_DR", "T2_Ast", "T2_TO", "T2_Stl", "T2_Blk", "T2_PF",
    "PointDiff",
]

In [16]:
def add_season_averages(regular_df: pd.DataFrame, tourney_df:pd.DataFrame):
    # 1. Identify the raw stats we want to average
    # We exclude IDs and Loc, but KEEP Score because Points Per Game is a great feature
    exclude_cols = ['T1_TeamID', 'T1_Loc']
    stat_cols = [col for col in regular_df.columns if col.startswith('T1_') and col not in exclude_cols]
    
    # 2. Calculate the end-of-season averages using ONLY regular season data
    # Because you doubled the data, T1_ represents every team's perspective perfectly
    season_averages = regular_df.groupby(['Season', 'T1_TeamID'])[stat_cols].mean().reset_index()
    
    # 3. Create T1 and T2 versions of these averages for easy merging
    t1_averages = season_averages.copy()
    t1_averages.columns = ['Season', 'T1_TeamID'] + [col.replace('T1_', 'T1_avg_') for col in stat_cols]
    
    t2_averages = season_averages.copy()
    t2_averages.columns = ['Season', 'T2_TeamID'] + [col.replace('T1_', 'T2_avg_') for col in stat_cols]
    
    # 4. Merge the averages back onto BOTH datasets
    def merge_and_clean(df):
        merged = pd.merge(df, t1_averages, on=['Season', 'T1_TeamID'], how='left')
        merged = pd.merge(merged, t2_averages, on=['Season', 'T2_TeamID'], how='left')
        
        # 5. CRITICAL: Drop the raw in-game box score stats so they don't leak!
        # We keep the targets (win, PointDiff) and IDs, but drop the raw FGM, AST, etc.
        raw_t1_stats = [col for col in stat_cols]
        raw_t2_stats = [col.replace('T1_', 'T2_') for col in stat_cols]
        merged = merged.drop(columns=raw_t1_stats + raw_t2_stats)
        
        return merged

    new_regular_df = merge_and_clean(regular_df)
    new_tourney_df = merge_and_clean(tourney_df)
    
    return new_regular_df, new_tourney_df

# Apply the function
regular_results_detailed, tourney_results_detailed = add_season_averages(regular_results_detailed, tourney_results_detailed)

regular_results_detailed

,Season,DayNum,T2_TeamID,T1_TeamID,NumOT,T2_Loc,T1_Loc,PointDiff,win,men_women,Seed_diff,T1_avg_Score,T1_avg_FGM,T1_avg_FGA,T1_avg_FGM3,T1_avg_FGA3,T1_avg_FTM,T1_avg_FTA,T1_avg_OR,T1_avg_DR,T1_avg_Ast,T1_avg_TO,T1_avg_Stl,T1_avg_Blk,T1_avg_PF,T1_avg_seed,T2_avg_Score,T2_avg_FGM,T2_avg_FGA,T2_avg_FGM3,T2_avg_FGA3,T2_avg_FTM,T2_avg_FTA,T2_avg_OR,T2_avg_DR,T2_avg_Ast,T2_avg_TO,T2_avg_Stl,T2_avg_Blk,T2_avg_PF,T2_avg_seed
0,2003,10,1328,1104,0,N,N,6,1,1,-9.0,69.285714,24.035714,57.178571,6.357143,19.857143,14.857143,20.928571,13.571429,23.928571,12.107143,13.285714,6.607143,3.785714,18.035714,10.0,71.166667,25.266667,56.533333,7.466667,18.966667,13.166667,18.600000,12.133333,24.966667,14.166667,11.800000,6.933333,3.766667,18.600000,1.0
1,2003,10,1393,1272,0,N,N,7,1,1,-4.0,74.517241,26.275862,60.000000,7.000000,20.068966,14.965517,22.896552,14.068966,25.965517,16.620690,13.793103,7.379310,5.068966,18.758621,7.0,80.103448,29.241379,62.206897,5.241379,15.862069,16.379310,23.620690,14.310345,26.896552,14.965517,13.620690,8.310345,7.275862,16.586207,3.0
2,2003,11,1437,1266,0,N,N,12,1,1,17.0,78.392857,27.214286,56.250000,5.785714,15.250000,18.178571,23.607143,13.107143,24.071429,16.321429,13.571429,6.000000,3.642857,18.642857,3.0,72.200000,24.833333,59.066667,6.666667,19.100000,15.866667,22.266667,14.700000,23.700000,13.066667,16.033333,7.500000,3.400000,20.900000,20.0
3,2003,11,1457,1296,0,N,N,6,1,1,0.0,69.612903,24.354839,53.064516,6.290323,16.419355,14.612903,22.387097,13.000000,22.645161,12.677419,17.000000,7.612903,3.612903,19.806452,20.0,69.428571,24.321429,56.285714,7.071429,20.107143,13.714286,21.571429,12.035714,23.964286,12.607143,14.642857,7.607143,5.392857,19.642857,20.0
4,2003,11,1208,1400,0,N,N,6,1,1,19.0,78.857143,28.000000,62.428571,5.857143,16.785714,17.000000,23.785714,16.178571,26.142857,14.500000,13.428571,6.392857,3.857143,20.357143,1.0,79.185185,28.518519,61.444444,6.703704,17.629630,15.444444,21.629630,12.814815,24.592593,17.925926,11.555556,7.629630,4.370370,17.185185,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423427,2026,131,3471,3218,0,N,N,-12,0,0,-6.0,64.161290,23.032258,54.645161,7.000000,20.129032,11.096774,14.903226,7.709677,25.870968,12.419355,15.548387,5.709677,6.354839,14.935484,20.0,70.580645,26.032258,61.645161,6.290323,20.064516,12.225806,17.064516,7.419355,22.645161,14.096774,12.483871,10.741935,2.838710,15.322581,14.0
423428,2026,132,3158,3220,0,N,N,-12,0,0,-6.0,53.656250,20.406250,55.187500,5.031250,16.937500,7.812500,11.531250,9.187500,20.156250,10.718750,14.906250,6.843750,2.187500,16.437500,20.0,71.000000,26.000000,65.633333,7.266667,22.333333,11.733333,15.966667,13.200000,22.333333,12.900000,11.900000,10.200000,1.900000,15.000000,14.0
423429,2026,132,3192,3254,0,H,A,-22,0,0,-5.0,69.700000,24.366667,57.900000,7.200000,20.800000,13.766667,18.833333,9.533333,23.633333,13.133333,15.833333,6.633333,4.100000,17.266667,20.0,67.875000,25.156250,59.656250,8.906250,25.875000,8.656250,11.968750,12.187500,24.468750,13.562500,12.187500,6.125000,3.281250,14.000000,15.0
423430,2026,132,3221,3250,0,H,A,-7,0,0,-5.0,66.387097,22.580645,52.322581,8.516129,25.774194,12.709677,16.451613,6.580645,19.032258,14.774194,14.774194,7.741935,1.516129,14.516129,20.0,61.312500,23.062500,55.875000,5.218750,18.531250,9.968750,13.281250,7.531250,25.312500,14.812500,13.812500,7.531250,3.468750,13.125000,15.0


In [17]:
regular_results_detailed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423432 entries, 0 to 423431
Data columns (total 41 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Season        423432 non-null  int64  
 1   DayNum        423432 non-null  int64  
 2   T2_TeamID     423432 non-null  int64  
 3   T1_TeamID     423432 non-null  int64  
 4   NumOT         423432 non-null  int64  
 5   T2_Loc        423432 non-null  object 
 6   T1_Loc        423432 non-null  object 
 7   PointDiff     423432 non-null  int64  
 8   win           423432 non-null  int64  
 9   men_women     423432 non-null  int64  
 10  Seed_diff     423432 non-null  float64
 11  T1_avg_Score  423432 non-null  float64
 12  T1_avg_FGM    423432 non-null  float64
 13  T1_avg_FGA    423432 non-null  float64
 14  T1_avg_FGM3   423432 non-null  float64
 15  T1_avg_FGA3   423432 non-null  float64
 16  T1_avg_FTM    423432 non-null  float64
 17  T1_avg_FTA    423432 non-null  float64
 18  T1_a

MODEL PREP

In [18]:
def prepare_tourney_focused_split(regular_df, tourney_df, val_start_season=2021, test_season=2025):
    # 1. Identify valid numeric features
    exclude_cols = ['Season', 'DayNum', 'T1_TeamID', 'T2_TeamID', 'T1_Loc', 'T2_Loc', 
                    'PointDiff', 'win', 'T1_seed', 'T2_seed', 'men_women']
    
    t1_cols = [col for col in regular_df.columns if col.startswith('T1_')]
    core_features = [col.replace('T1_', '') for col in t1_cols if col not in exclude_cols]
    
    def calculate_differences(df):
        diff_df = pd.DataFrame()
        diff_df['Season'] = df['Season']
        diff_df['win'] = df['win']
        diff_df['Seed_diff'] = df['Seed_diff'].fillna(0)
        diff_df['men_women'] = df['men_women']
        
        for feature in core_features:
            if f'T1_{feature}' in df.columns and f'T2_{feature}' in df.columns:
                diff_df[f'diff_{feature}'] = df[f'T1_{feature}'] - df[f'T2_{feature}']
        return diff_df

    # Process differences separately before mixing
    reg_diffs = calculate_differences(regular_df)
    tourney_diffs = calculate_differences(tourney_df)
    
    feature_cols = [col for col in reg_diffs.columns if col not in ['Season', 'win']]
    
    # 2. THE NEW SPLIT LOGIC
    
    # TRAIN: All regular season games before 2025 + older tournament games (before 2021)
    train_reg = reg_diffs[reg_diffs['Season'] < test_season]
    train_tourney = tourney_diffs[tourney_diffs['Season'] < val_start_season]
    train_df = pd.concat([train_reg, train_tourney], axis=0, ignore_index=True)
    
    # VAL: ONLY Tournament games from recent seasons (e.g., 2021, 2022, 2023, 2024)
    # This ensures early stopping optimizes for the tournament!
    val_df = tourney_diffs[(tourney_diffs['Season'] >= val_start_season) & 
                           (tourney_diffs['Season'] < test_season)].copy()
    
    # TEST: ONLY 2025 Tournament games
    test_df = tourney_diffs[tourney_diffs['Season'] == test_season].copy()
    
    # 3. Scale the Data (Fit strictly on TRAIN)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_df[feature_cols])
    X_val_scaled = scaler.transform(val_df[feature_cols])
    X_test_scaled = scaler.transform(test_df[feature_cols])
    
    # 4. Create PyTorch DataLoaders
    def to_loader(X, y, batch_size, shuffle=False):
        tensor_X = torch.FloatTensor(X)
        tensor_y = torch.FloatTensor(y.values).unsqueeze(1)
        return DataLoader(TensorDataset(tensor_X, tensor_y), batch_size=batch_size, shuffle=shuffle)
    
    loaders = {
        # High batch size for train because it has ~400k rows
        'train': to_loader(X_train_scaled, train_df['win'], batch_size=512, shuffle=True),
        # Lower batch size for val/test since they are small (~260 and ~67 rows)
        'val': to_loader(X_val_scaled, val_df['win'], batch_size=64, shuffle=False),
        'test': to_loader(X_test_scaled, test_df['win'], batch_size=64, shuffle=False),
        'input_dim': len(feature_cols),
        'scaler': scaler 
    }
    
    print(f"Train Shape (Reg + Old Tourney): {X_train_scaled.shape}")
    print(f"Val Shape ({val_start_season}-{test_season-1} Tourney Only): {X_val_scaled.shape}")
    print(f"Test Shape ({test_season} Tourney Only): {X_test_scaled.shape}")
    
    return loaders

# Execute
data_splits = prepare_tourney_focused_split(regular_results_detailed, tourney_results_detailed)

Train Shape (Reg + Old Tourney): (382500, 17)
Val Shape (2021-2024 Tourney Only): (1062, 17)
Test Shape (2025 Tourney Only): (268, 17)


# MODEL

In [19]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),     
            nn.ReLU(),
            nn.Dropout(CFG['dropout']),        
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(CFG['dropout']),
            nn.Linear(32, 1),
            #nn.Sigmoid() # Outputs a probability between 0 and 1
        )

    def forward(self, x):
        return self.network(x)

In [20]:
DATA['sample_sub2']

,ID,Pred
0,2026_1101_1102,0.5
1,2026_1101_1103,0.5
2,2026_1101_1104,0.5
3,2026_1101_1105,0.5
4,2026_1101_1106,0.5
...,...,...
132128,2026_3478_3480,0.5
132129,2026_3478_3481,0.5
132130,2026_3479_3480,0.5
132131,2026_3479_3481,0.5


# TRAIN

In [21]:
model = MLP(input_dim=data_splits['input_dim']).to(DEVICE)
criterion = nn.BCEWithLogitsLoss() 
optimizer = Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

best_val_brier = float('inf') 
epochs_no_improve = 0
best_model_state = None

print("Starting Single-Run Training...")
print("-" * 50)

for epoch in range(CFG['epochs']):
    model.train()
    train_loss = 0.0
    
    # --- Training Pass ---
    for batch_X, batch_y in data_splits['train']:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        
        # Apply Label Smoothing to targets
        smoothed_y = batch_y * (1.0 - CFG['label_smoothing']) + 0.5 * CFG['label_smoothing']
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, smoothed_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # --- Validation Pass ---
    model.eval()
    val_loss = 0.0
    val_brier = 0.0           
    total_val_samples = 0     
    
    with torch.no_grad():
        for batch_X, batch_y in data_splits['val']:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            predictions = model(batch_X)
            
            # Standard BCE Loss for information
            loss = criterion(predictions, batch_y) 
            val_loss += loss.item()
            
            # Brier Score Calculation (MSE of probabilities)
            probs = torch.sigmoid(predictions)
            val_brier += torch.sum((probs - batch_y) ** 2).item()
            total_val_samples += batch_y.size(0)
            
    # Calculate Epoch Averages
    train_loss /= len(data_splits['train'])
    val_loss /= len(data_splits['val'])
    val_brier /= total_val_samples 
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss (BCE): {train_loss:.4f} | Val Loss (BCE): {val_loss:.4f} | Val Brier: {val_brier:.4f}")
    
    # --- Early Stopping (Optimizing for Brier) ---
    if val_brier < best_val_brier:
        best_val_brier = val_brier
        epochs_no_improve = 0
        # THE FIX: Deep clone the tensors so they detach from the actively training model
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= CFG['patience']:
            print(f"Early stopping triggered at epoch {epoch}. Best Val Brier: {best_val_brier:.4f}")
            break

# Restore best weights
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    
print("-" * 50)
print("Training Complete. Model loaded with best validation weights.")

Starting Single-Run Training...
--------------------------------------------------
Epoch   0 | Train Loss (BCE): 0.5534 | Val Loss (BCE): 0.5495 | Val Brier: 0.1863
Epoch  10 | Train Loss (BCE): 0.5409 | Val Loss (BCE): 0.5439 | Val Brier: 0.1832
Epoch  20 | Train Loss (BCE): 0.5406 | Val Loss (BCE): 0.5452 | Val Brier: 0.1835
Epoch  30 | Train Loss (BCE): 0.5408 | Val Loss (BCE): 0.5434 | Val Brier: 0.1830
Early stopping triggered at epoch 33. Best Val Brier: 0.1824
--------------------------------------------------
Training Complete. Model loaded with best validation weights.


In [22]:
# --- EVALUATE ON 2025 HOLDOUT ---
model.eval()
test_brier = 0.0
total_test_samples = 0

with torch.no_grad():
    for batch_X, batch_y in data_splits['test']:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        
        predictions = model(batch_X)
        probs = torch.sigmoid(predictions)
        
        test_brier += torch.sum((probs - batch_y) ** 2).item()
        total_test_samples += batch_y.size(0)

test_brier /= total_test_samples

print("=" * 50)
print(f"🏀 FINAL HOLDOUT SCORE (2025 Tourney) Brier: {test_brier:.4f} 🏀")
print("=" * 50)

🏀 FINAL HOLDOUT SCORE (2025 Tourney) Brier: 0.1449 🏀


In [23]:
def prepare_tourney_focused_split(regular_df, tourney_df, val_start_season=2021, test_season=2026):
    print(f"Preparing data splits (Test Season: {test_season})...")
    
    # 1. Identify valid numeric features
    exclude_cols = ['Season', 'DayNum', 'T1_TeamID', 'T2_TeamID', 'T1_Loc', 'T2_Loc', 
                    'PointDiff', 'win', 'T1_seed', 'T2_seed', 'men_women']
    
    t1_cols = [col for col in regular_df.columns if col.startswith('T1_')]
    core_features = [col.replace('T1_', '') for col in t1_cols if col not in exclude_cols]
    
    def calculate_differences(df):
        diff_df = pd.DataFrame()
        diff_df['Season'] = df['Season']
        diff_df['win'] = df['win']
        # fillna(0) removed here because seed_transform already assigned our UNSEEDED_VAL
        diff_df['Seed_diff'] = df['Seed_diff'] 
        diff_df['men_women'] = df['men_women']
        
        for feature in core_features:
            if f'T1_{feature}' in df.columns and f'T2_{feature}' in df.columns:
                diff_df[f'diff_{feature}'] = df[f'T1_{feature}'] - df[f'T2_{feature}']
        return diff_df

    # Process differences separately before mixing
    reg_diffs = calculate_differences(regular_df)
    tourney_diffs = calculate_differences(tourney_df)
    
    feature_cols = [col for col in reg_diffs.columns if col not in ['Season', 'win']]
    
    # 2. THE SPLIT LOGIC
    # TRAIN: All regular season games UP TO AND INCLUDING the test_season (<=), 
    # plus historical tournament games (before val_start_season)
    train_reg = reg_diffs[reg_diffs['Season'] <= test_season] 
    train_tourney = tourney_diffs[tourney_diffs['Season'] < val_start_season]
    train_df = pd.concat([train_reg, train_tourney], axis=0, ignore_index=True)
    
    # VAL: ONLY Tournament games from the validation window
    val_df = tourney_diffs[(tourney_diffs['Season'] >= val_start_season) & 
                           (tourney_diffs['Season'] < test_season)].copy()
    
    # TEST: ONLY the target test season's Tournament games (Will be empty for 2026 initially!)
    test_df = tourney_diffs[tourney_diffs['Season'] == test_season].copy()
    
    # 3. Scale the Data (Fit strictly on TRAIN)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_df[feature_cols])
    X_val_scaled = scaler.transform(val_df[feature_cols])
    
    # Handle the empty test set gracefully
    if len(test_df) > 0:
        X_test_scaled = scaler.transform(test_df[feature_cols])
    else:
        X_test_scaled = np.array([])
    
    # 4. Create PyTorch DataLoaders
    def to_loader(X, y, batch_size, shuffle=False):
        if len(X) == 0:
            return [] # Return empty list if no data (e.g., 2026 test set)
        tensor_X = torch.FloatTensor(X)
        tensor_y = torch.FloatTensor(y.values).unsqueeze(1)
        return DataLoader(TensorDataset(tensor_X, tensor_y), batch_size=batch_size, shuffle=shuffle)
    
    loaders = {
        'train': to_loader(X_train_scaled, train_df['win'], batch_size=512, shuffle=True),
        'val': to_loader(X_val_scaled, val_df['win'], batch_size=64, shuffle=False),
        'test': to_loader(X_test_scaled, test_df['win'] if len(test_df) > 0 else pd.Series([]), batch_size=64, shuffle=False),
        'input_dim': len(feature_cols),
        'scaler': scaler 
    }
    
    print(f"Train Shape: {X_train_scaled.shape}")
    print(f"Val Shape:   {X_val_scaled.shape}")
    print(f"Test Shape:  {X_test_scaled.shape}")
    
    return loaders

In [24]:
def generate_submission(model, scaler, regular_df, sample_sub_df):
    print("Preparing Submission Data...")
    sub = sample_sub_df.copy()
    
    # 1. Parse the Kaggle ID string
    sub['Season'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
    sub['T1_TeamID'] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
    sub['T2_TeamID'] = sub['ID'].apply(lambda x: int(x.split('_')[2]))
    
    # 2. Add men_women flag (1 if ID starts with 1, else 0)
    sub['men_women'] = sub['T1_TeamID'].apply(lambda x: 1 if str(x).startswith('1') else 0)
    
    # 3. Add Seeds (using the function we fixed earlier with the ghost seed)
    sub = seed_transform(sub)
    
    # 4. Extract unique season averages from our regular season dataset
    avg_cols = [c for c in regular_df.columns if c.startswith('T1_avg_')]
    season_averages = regular_df[['Season', 'T1_TeamID'] + avg_cols].drop_duplicates()
    
    # 5. Merge averages for T1
    sub = pd.merge(sub, season_averages, on=['Season', 'T1_TeamID'], how='left')
    
    # 6. Create T2 averages mapping and merge for T2
    season_averages_t2 = season_averages.copy()
    season_averages_t2.columns = ['Season', 'T2_TeamID'] + [c.replace('T1_avg_', 'T2_avg_') for c in avg_cols]
    sub = pd.merge(sub, season_averages_t2, on=['Season', 'T2_TeamID'], how='left')
    
    # 7. DYNAMIC DIFFERENCE CALCULATION
    # Read exactly what the scaler was trained on to prevent KeyErrors
    feature_cols = list(scaler.feature_names_in_)
    
    for col in feature_cols:
        if col.startswith('diff_'):
            base_feat = col.replace('diff_', '') # e.g., 'avg_Score'
            t1_col = f'T1_{base_feat}'           # e.g., 'T1_avg_Score'
            t2_col = f'T2_{base_feat}'           # e.g., 'T2_avg_Score'
            sub[col] = sub[t1_col] - sub[t2_col]
            
    # Check for missing averages (happens if a team had no D1 regular season games)
    if sub[feature_cols].isnull().any().any():
        print("Warning: Missing averages found in submission set. Filling with 0.")
        sub[feature_cols] = sub[feature_cols].fillna(0)
        
    # 8. Scale the features using the fitted training scaler
    X_sub_scaled = scaler.transform(sub[feature_cols])
    
    # 9. Predict using the neural network
    print("Running neural network predictions...")
    model.eval()
    tensor_X = torch.FloatTensor(X_sub_scaled).to(DEVICE)
    
    with torch.no_grad():
        predictions = model(tensor_X)
        probs = torch.sigmoid(predictions).cpu().numpy().squeeze()
        
    # 10. Clip probabilities to avoid infinite log loss penalties
    sub["Pred"] = np.clip(probs, 1e-5, 1 - 1e-5)
    
    # 11. Format final output
    submission = sub[["ID", "Pred"]]
    submission.to_csv("submission.csv", index=False)
    
    print(f"Success! Submission saved to 'submission.csv'. Shape: {submission.shape}")
    return submission

In [25]:
# ==============================================================================
# FINAL EXECUTION BLOCK: DATA PREP, ENSEMBLE TRAINING, AND SUBMISSION
# ==============================================================================

print("1. Preparing Production Data for 2026...")
# This uses the <= fix we discussed to include the 2026 regular season
data_splits_prod = prepare_tourney_focused_split(
    regular_results_detailed, 
    tourney_results_detailed, 
    val_start_season=2022, 
    test_season=2026 
)

# Setup our Seed Ensemble
random_seeds = [42, 100, 2026]
all_preds = []
criterion = nn.BCEWithLogitsLoss() 

for s in random_seeds:
    print(f"\n{'='*40}")
    print(f"--- Training Production Model with Seed {s} ---")
    print(f"{'='*40}")
    
    # 2. Lock in the seed for this specific run
    torch.manual_seed(s)
    np.random.seed(s)
    
    # 3. Initialize a fresh model and optimizer
    final_model = MLP(input_dim=data_splits_prod['input_dim']).to(DEVICE)
    optimizer = Adam(final_model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    
    best_val_brier = float('inf') 
    epochs_no_improve = 0
    best_model_state = None
    
    # 4. Training Loop with Brier Score Early Stopping
    for epoch in range(CFG['epochs']):
        final_model.train()
        
        for batch_X, batch_y in data_splits_prod['train']:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            smoothed_y = batch_y * (1.0 - CFG['label_smoothing']) + 0.5 * CFG['label_smoothing']
            
            optimizer.zero_grad()
            predictions = final_model(batch_X)
            loss = criterion(predictions, smoothed_y)
            loss.backward()
            optimizer.step()
            
        # Validation Pass
        final_model.eval()
        val_brier = 0.0           
        total_val_samples = 0     
        
        with torch.no_grad():
            for batch_X, batch_y in data_splits_prod['val']:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                predictions = final_model(batch_X)
                probs = torch.sigmoid(predictions)
                val_brier += torch.sum((probs - batch_y) ** 2).item()
                total_val_samples += batch_y.size(0)
                
        val_brier /= total_val_samples 
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Val Brier: {val_brier:.4f}")
        
        if val_brier < best_val_brier:
            best_val_brier = val_brier
            epochs_no_improve = 0
            best_model_state = {k: v.cpu().clone() for k, v in final_model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CFG['patience']:
                print(f"Early stopping at epoch {epoch}. Best Val Brier: {best_val_brier:.4f}")
                break
                
    # 5. Load the best weights for this specific seed
    if best_model_state is not None:
        final_model.load_state_dict(best_model_state)
        
    # 6. Generate predictions for SampleSubmissionStage2
    print(f"\nGenerating Stage 2 predictions for Seed {s}...")
    sub = generate_submission(
        model=final_model, 
        scaler=data_splits_prod['scaler'], 
        regular_df=regular_results_detailed, 
        sample_sub_df=DATA['sample_sub2']
    )
    
    # Store the predicted probabilities
    all_preds.append(sub['Pred'].values)


# 7. Finalize the Ensemble
print("\n" + "=" * 50)
print("🏀 Averaging predictions for final ensemble... 🏀")
final_submission = DATA['sample_sub2'].copy()
final_submission['Pred'] = np.mean(all_preds, axis=0)

# Save the ultimate file
final_submission.to_csv("submission.csv", index=False)
print("Success! Ensembled submission saved to 'submission.csv'.")
print("=" * 50)

1. Preparing Production Data for 2026...
Preparing data splits (Test Season: 2026)...
Train Shape: (427180, 17)
Val Shape:   (1072, 17)
Test Shape:  (0,)

--- Training Production Model with Seed 42 ---
Epoch   0 | Val Brier: 0.1769
Epoch  10 | Val Brier: 0.1719
Epoch  20 | Val Brier: 0.1725
Epoch  30 | Val Brier: 0.1718
Epoch  40 | Val Brier: 0.1710
Early stopping at epoch 41. Best Val Brier: 0.1708

Generating Stage 2 predictions for Seed 42...
Preparing Submission Data...
Running neural network predictions...
Success! Submission saved to 'submission.csv'. Shape: (132133, 2)

--- Training Production Model with Seed 100 ---
Epoch   0 | Val Brier: 0.1762
Epoch  10 | Val Brier: 0.1728
Epoch  20 | Val Brier: 0.1723
Early stopping at epoch 29. Best Val Brier: 0.1708

Generating Stage 2 predictions for Seed 100...
Preparing Submission Data...
Running neural network predictions...
Success! Submission saved to 'submission.csv'. Shape: (132133, 2)

--- Training Production Model with Seed 2026 